# Downsampling a signal with `resample`

`resample` buckets a stream of values into causal windows and reduces each
bucket to one or more summary values. There are two bucketing modes:

- `every=n` -- fixed index interval (e.g. every 25 samples).
- `count=k` -- fixed number of samples per bucket.

The reduction is controlled by `agg`, which can be a built-in string such as
`"mean"`, `"sum"`, or `"ohlc"`, any screamer functor, or a dict
`{name: agg, ...}` that produces several named columns at once.

This notebook uses a generic noisy signal (slow sine wave plus broadband
noise) to illustrate downsampling and per-bucket noise estimation. For
financial tick-bar usage see
[Tick bars and custom aggregations](10-tick-bars-and-custom-aggregations.ipynb).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from screamer import ExpandingStd
from screamer.streams import resample

SEED = 1
rng = np.random.default_rng(SEED)

# Noisy signal: slow sine wave plus broadband noise.
t   = np.arange(500)
sig = np.sin(t / 40) + rng.normal(scale=0.4, size=500)

## Downsampling with `agg="mean"`

Each bucket covers 25 consecutive samples. `agg="mean"` reduces every
bucket to its sample mean. The result is a `Stream` that unpacks as
`(values, index)`, where `index` holds the left-edge label of each bucket.

In [ ]:
EVERY = 25
bars_mean, bar_idx = resample(sig, t, every=EVERY, agg="mean")

fig, ax = plt.subplots(figsize=(9, 3.5))

# Raw signal in light grey.
ax.plot(t, sig, lw=0.5, color="0.75", label="raw signal")

# Downsampled bar means as horizontal steps.
ax.step(bar_idx, bars_mean, where="post", lw=2.0,
        color="steelblue", label=f"bucket mean (every={EVERY})")
ax.scatter(bar_idx, bars_mean, s=30, color="steelblue", zorder=3)

ax.set_xlabel("sample index")
ax.set_title("Raw signal vs. downsampled bucket means")
ax.legend(fontsize=9)
plt.tight_layout()

## Per-bucket noise level with dict `agg`

Passing `agg` as a dict runs multiple reducers over the same bucketing and
returns a labelled `Stream` with one column per entry. Here the bucket mean
and per-bucket noise (standard deviation) are computed together. The
`ExpandingStd()` functor is reset automatically at each bucket boundary, so
it measures the spread of values *within* each bucket rather than globally.

The noise column tracks how much spread is in each bucket: high-noise
buckets carry less reliable mean estimates.

In [ ]:
bars = resample(
    sig, t,
    every=EVERY,
    agg={"mean": "mean", "noise": ExpandingStd()},
)

print("columns:", bars.columns)

fig, (ax_sig, ax_noise) = plt.subplots(2, 1, figsize=(9, 5), sharex=True)

ax_sig.plot(t, sig, lw=0.5, color="0.75", label="raw signal")
ax_sig.step(bars.index, bars["mean"], where="post",
            lw=2.0, color="steelblue", label="bucket mean")
ax_sig.legend(fontsize=9)
ax_sig.set_ylabel("signal")
ax_sig.set_title("Bucket mean and per-bucket noise estimate")

ax_noise.step(bars.index, bars["noise"], where="post",
              lw=1.8, color="darkorange", label="bucket std (noise)")
ax_noise.scatter(bars.index, bars["noise"], s=30, color="darkorange", zorder=3)
ax_noise.set_ylabel("std")
ax_noise.set_xlabel("bar left edge")
ax_noise.legend(fontsize=9)

plt.tight_layout()

The same bucketing works with `agg="ohlc"` to produce open/high/low/close
summaries of each bucket, which is useful when the index represents time and
you want to capture the price range inside each bar.